# Análise dos Resultados

Lê `resultados/medicoes.csv` e `resultados/medicoes_poda.csv`, computa resumos estatísticos e gera todos os gráficos comparativos.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 220, "font.size": 12})

MEDICOES_PATH      = Path("resultados/medicoes.csv")
MEDICOES_PODA_PATH = Path("resultados/medicoes_poda.csv")

## 1. Carregamento dos dados

In [ ]:
def carregar(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["estourou_timeout"] = df["estourou_timeout"].astype(str).str.lower() == "true"
    return df

df      = carregar(MEDICOES_PATH)
df_poda = carregar(MEDICOES_PODA_PATH)

print(f"medicoes.csv      : {len(df)} linhas")
print(f"medicoes_poda.csv : {len(df_poda)} linhas")
df.head()

## 2. Resumo estatístico

Agrega as repetições em **média + desvio-padrão** por `(algoritmo, tipo_instancia, n)`.

In [ ]:
def resumir(df: pd.DataFrame) -> pd.DataFrame:
    validas = df[~df["estourou_timeout"]]
    grp = validas.groupby(["algoritmo", "tipo_instancia", "n"])
    resumo = grp.agg(
        execucoes_validas=("repeticao", "count"),
        tempo_medio_s=("tempo_s", "mean"),
        tempo_desvio_s=("tempo_s", "std"),
        memoria_media_mb=("pico_memoria_bytes", lambda x: x.mean() / 1e6),
        memoria_desvio_mb=("pico_memoria_bytes", lambda x: x.std() / 1e6),
        estados_medios=("estados", "mean"),
    ).reset_index()
    timeouts = (
        df[df["estourou_timeout"]]
        .groupby(["algoritmo", "tipo_instancia", "n"])
        .size()
        .reset_index(name="timeouts")
    )
    resumo = resumo.merge(timeouts, on=["algoritmo", "tipo_instancia", "n"], how="left")
    resumo["timeouts"] = resumo["timeouts"].fillna(0).astype(int)
    return resumo

resumo      = resumir(df)
resumo_poda = resumir(df_poda)

## 3. Configurações visuais

In [ ]:
CORES = {
    "backtracking":           "#1f77b4",
    "meet_in_middle":         "#d62728",
    "backtracking_com_poda":  "#2ca02c",
    "backtracking_sem_poda":  "#7f7f7f",
}
ROTULOS = {
    "backtracking":           "Backtracking (com poda)",
    "meet_in_middle":         "Divisão e Conquista (MITM)",
    "backtracking_com_poda":  "Backtracking com poda",
    "backtracking_sem_poda":  "Backtracking sem poda",
}
ROTULOS_TIPO = {
    "com_solucao":  "com solução",
    "sem_solucao":  "sem solução",
    "alvo_pequeno": "alvo pequeno",
    "alvo_grande":  "alvo grande",
}
MARCADORES = {
    "backtracking":           "o",
    "meet_in_middle":         "s",
    "backtracking_com_poda":  "^",
    "backtracking_sem_poda":  "v",
}

## 4. Tempo médio vs n por tipo de instância

In [ ]:
tipos = sorted(resumo["tipo_instancia"].unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

for ax, tipo in zip(axes, tipos):
    sub = resumo[resumo["tipo_instancia"] == tipo]
    for alg, grupo in sub.groupby("algoritmo"):
        grupo = grupo.sort_values("n")
        ax.errorbar(
            grupo["n"], grupo["tempo_medio_s"],
            yerr=grupo["tempo_desvio_s"].fillna(0),
            fmt=f"{MARCADORES.get(alg, 'o')}-",
            color=CORES.get(alg),
            label=ROTULOS.get(alg, alg),
            capsize=3,
        )
    ax.set_yscale("log")
    ax.set_title(f"Instâncias {ROTULOS_TIPO.get(tipo, tipo)}")
    ax.set_xlabel("n (número de transações)")
    ax.set_ylabel("tempo médio (s)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("Tempo médio vs n – Backtracking vs. Meet in the Middle", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 5. Pico de memória vs n por tipo de instância

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

for ax, tipo in zip(axes, tipos):
    sub = resumo[resumo["tipo_instancia"] == tipo]
    for alg, grupo in sub.groupby("algoritmo"):
        grupo = grupo.sort_values("n")
        ax.errorbar(
            grupo["n"], grupo["memoria_media_mb"],
            yerr=grupo["memoria_desvio_mb"].fillna(0),
            fmt=f"{MARCADORES.get(alg, 's')}-",
            color=CORES.get(alg),
            label=ROTULOS.get(alg, alg),
            capsize=3,
        )
    ax.set_yscale("log")
    ax.set_title(f"Instâncias {ROTULOS_TIPO.get(tipo, tipo)}")
    ax.set_xlabel("n (número de transações)")
    ax.set_ylabel("pico de memória (MB)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("Pico de memória vs n – Backtracking vs. Meet in the Middle", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Efeito da poda no Backtracking (estados explorados)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Estados explorados
sub = resumo_poda[resumo_poda["tipo_instancia"] == "com_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.plot(
        grupo["n"], grupo["estados_medios"],
        f"{MARCADORES.get(alg, '^')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("nós visitados (média)")
ax.set_title("Estados explorados – com poda vs. sem poda")
ax.grid(True, which="both", alpha=0.3)
ax.legend()


fig.tight_layout()
plt.show()

## 7. Sensibilidade do Backtracking à entrada

In [ ]:
ESTILOS_TIPO = {
    "com_solucao":  ("o-",  "com solução"),
    "sem_solucao":  ("x--", "sem solução"),
    "alvo_pequeno": ("v:",  "alvo pequeno"),
    "alvo_grande":  ("d-.", "alvo grande"),
}

fig, ax = plt.subplots(figsize=(9, 5))

# Tempo
sub_bt = resumo[resumo["algoritmo"] == "backtracking"]
for tipo, (estilo, rotulo) in ESTILOS_TIPO.items():
    grupo = sub_bt[sub_bt["tipo_instancia"] == tipo].sort_values("n")
    if grupo.empty:
        continue
    ax.plot(grupo["n"], grupo["tempo_medio_s"], estilo, label=rotulo)
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("tempo médio (s)")
ax.set_title("Backtracking – tempo por tipo de instância")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## 8. Comparativo direto: memoria e tempo por algoritmo

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = resumo[resumo["tipo_instancia"] == "com_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.errorbar(
        grupo["n"], grupo["memoria_media_mb"],
        yerr=grupo["memoria_desvio_mb"].fillna(0),
        fmt=f"{MARCADORES.get(alg, 'o')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
        capsize=4,
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("pico de memória (MB)")
ax.set_title("Backtracking vs. Meet in the Middle – instâncias com solução")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = resumo[resumo["tipo_instancia"] == "sem_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.errorbar(
        grupo["n"], grupo["tempo_medio_s"],
        yerr=grupo["tempo_desvio_s"].fillna(0),
        fmt=f"{MARCADORES.get(alg, 'o')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
        capsize=4,
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("tempo médio (s) – escala log")
ax.set_title("Backtracking vs. Meet in the Middle – instâncias sem solução")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Fator de redução da poda

In [ ]:
pivot_estados = resumo_poda.pivot_table(
    index="n",
    columns="algoritmo",
    values="estados_medios",
).dropna()

pivot_estados["fator_reducao"] = (
    pivot_estados["backtracking_sem_poda"] / pivot_estados["backtracking"]
)
print("Fator de redução de estados (sem poda / com poda):")
display(pivot_estados[["backtracking", "backtracking_sem_poda", "fator_reducao"]].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(pivot_estados.index.astype(str), pivot_estados["fator_reducao"], color="#2ca02c", alpha=0.8)
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("fator de redução (vs)")
ax.set_title("Quantas vezes a poda reduz os estados explorados")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 11. Resumo consolidado para o artigo (`artigo/artigo.tex`)

Recalcula, a partir dos CSVs atuais, **todas as métricas numéricas citadas no artigo**, agrupadas pelo trecho em que aparecem. Após regenerar `resultados/*.csv`, basta rodar esta célula e conferir cada bloco contra o texto:

| Bloco | Onde aparece no artigo |
|---|---|
| Metodologia | Seção 4.3 (n avaliados, repetições, tempo-limite) |
| Tabela 1 | `tab:sintese` (tempos com/sem solução + memória) |
| Experimento 1 | Seção 5.1 (acelerações do MITM, fatores de crescimento) — reutilizado no resumo e na conclusão |
| Experimento 2 | Seção 5.2 (pico do BT em KiB, pico do MITM, ordens de grandeza, projeção n=50) e `tab:eixos` |
| Experimento 3 | Seção 5.3 (nós com/sem poda, fator máximo de redução) — reutilizado no resumo |
| Experimento 4 | Seção 5.4 (alvo pequeno, pico/estabilização do BT, dispersão entre classes, estabilidade do MITM, transição de densidade) |
| Linhas de código | Seção 5.5 (facilidade de implementação) |

⚠️ `TIMEOUT_S` e a faixa de valores do gerador são constantes replicadas nesta célula — mantenha em sincronia com `experimentos.ipynb` e `utils/gerador.py`.

In [ ]:
import ast
from math import log10
from statistics import geometric_mean

TIMEOUT_S = 45
VALOR_MIN_CENT, VALOR_MAX_CENT = 1_000, 5_000_000

BT, MITM, BT_SEM_PODA = "backtracking", "meet_in_middle", "backtracking_sem_poda"
TIPOS = ["com_solucao", "sem_solucao", "alvo_pequeno", "alvo_grande"]
NS = sorted(df["n"].unique())


def stats(dframe, alg, tipo, n):
    """Média das repetições válidas + timeouts de um ponto (algoritmo, tipo, n)."""
    sel = dframe[(dframe["algoritmo"] == alg)
                 & (dframe["tipo_instancia"] == tipo)
                 & (dframe["n"] == n)]
    val = sel[~sel["estourou_timeout"]]
    return {
        "tempo": val["tempo_s"].mean(),
        "mem_mb": val["pico_memoria_bytes"].mean() / 1e6,
        "estados": val["estados"].mean(),
        "timeouts": int(sel["estourou_timeout"].sum()),
        "total": len(sel),
    }


def eh_tl(st):
    """True quando todas as repetições do ponto estouraram o tempo-limite."""
    return st["total"] > 0 and st["timeouts"] == st["total"]


def fmt_t(t):
    if pd.isna(t):
        return "—"
    return f"{t:.3f} s" if t >= 5e-4 else f"{t:.1e} s"


def fmt_tempo(st):
    if st["total"] == 0:
        return "—"
    if eh_tl(st):
        return "T.L."
    txt = fmt_t(st["tempo"])
    if st["timeouts"]:
        txt += f" ({st['timeouts']}/{st['total']} T.L.)"
    return txt


def fmt_mem(mb):
    if pd.isna(mb):
        return "—"
    if mb < 0.1:
        return f"{mb * 1e6 / 1024:.1f} KiB"
    if mb < 1000:
        return f"{mb:.2f} MB"
    return f"{mb / 1000:.2f} GB"


def fatores_crescimento(dframe, alg, tipo, chave):
    """Fatores v(n)/v(n-5) entre pontos consecutivos sem timeout total."""
    st = {n: stats(dframe, alg, tipo, n) for n in NS}
    return [(b, st[b][chave] / st[a][chave])
            for a, b in zip(NS, NS[1:])
            if st[a]["total"] and st[b]["total"]
            and not eh_tl(st[a]) and not eh_tl(st[b])
            and not pd.isna(st[a][chave]) and st[a][chave] > 0
            and not pd.isna(st[b][chave])]


def secao(titulo):
    print()
    print("=" * 78)
    print(titulo)
    print("=" * 78)


# ------------------------------------------------------------------
secao("METODOLOGIA (Seção 4.3) — parâmetros do protocolo")
reps = df.groupby(["algoritmo", "tipo_instancia", "n"]).size()
print(f"Tamanhos n avaliados      : {list(NS)}")
print(f"Repetições por ponto      : {reps.min()}"
      + (f"–{reps.max()}" if reps.min() != reps.max() else "")
      + "  (o artigo cita esse número na Seção 4.3 e na legenda da Tabela 1)")
print(f"Tempo-limite por execução : {TIMEOUT_S} s")
print(f"Classes de instância      : {sorted(df['tipo_instancia'].unique())}")

# ------------------------------------------------------------------
secao("TABELA 1 (tab:sintese) — tempo médio (s) e pico de memória por n")
print(f"{'n':>3} | {'BT com sol.':>18} | {'MITM com sol.':>18} | "
      f"{'BT sem sol.':>18} | {'MITM sem sol.':>18} | memória BT / MITM")
for n in [x for x in NS]:
    bc, mc = stats(df, BT, "com_solucao", n), stats(df, MITM, "com_solucao", n)
    bs, ms = stats(df, BT, "sem_solucao", n), stats(df, MITM, "sem_solucao", n)
    print(f"{n:>3} | {fmt_tempo(bc):>18} | {fmt_tempo(mc):>18} | "
          f"{fmt_tempo(bs):>18} | {fmt_tempo(ms):>18} | "
          f"{fmt_mem(bc['mem_mb'])} / {fmt_mem(mc['mem_mb'])}")

# ------------------------------------------------------------------
secao("EXPERIMENTO 1 (Seção 5.1) — tempo nas instâncias sem solução")
print("Aceleração do MITM sobre o Backtracking (tempo BT / tempo MITM):")
for n in NS:
    b, m = stats(df, BT, "sem_solucao", n), stats(df, MITM, "sem_solucao", n)
    if not b["total"] or not m["total"]:
        continue
    if eh_tl(b) and eh_tl(m):
        print(f"  n={n:>2}: ambos T.L.")
    elif eh_tl(b):
        print(f"  n={n:>2}: BT = T.L., MITM = {fmt_t(m['tempo'])}")
    else:
        print(f"  n={n:>2}: BT = {fmt_t(b['tempo'])}, MITM = {fmt_t(m['tempo'])}"
              f"  →  {b['tempo'] / m['tempo']:.0f}x")
for alg, nome in [(BT, "Backtracking"), (MITM, "MITM")]:
    fs = fatores_crescimento(df, alg, "sem_solucao", "tempo")
    txt = ", ".join(f"{a - 5}→{a}: {f:.1f}x" for a, f in fs)
    regime = [f for a, f in fs if a >= 25]
    gm = f"  |  média geométrica (n≥25): {geometric_mean(regime):.1f}x" if regime else ""
    print(f"Fator de crescimento do tempo a cada +5 em n — {nome}:\n  {txt}{gm}")

# ------------------------------------------------------------------
secao("EXPERIMENTO 2 (Seção 5.2) — memória nas instâncias com solução")
bruto_bt = df[(df["algoritmo"] == BT) & (df["tipo_instancia"] == "com_solucao")]
print(f"Pico máximo do BT em todas as execuções (até n={bruto_bt['n'].max()}): "
      f"{bruto_bt['pico_memoria_bytes'].max() / 1024:.1f} KiB")
st_mem = {n: stats(df, MITM, "com_solucao", n) for n in NS}
com_mem = [n for n in NS if not pd.isna(st_mem[n]["mem_mb"])]
n_top = max(com_mem)
bt_top = stats(df, BT, "com_solucao", n_top)
razao = st_mem[n_top]["mem_mb"] / bt_top["mem_mb"]
print(f"Pico do MITM no maior n medido (n={n_top}): {fmt_mem(st_mem[n_top]['mem_mb'])}")
print(f"Razão MITM/BT em n={n_top}: {razao:.2e}  (~{log10(razao):.1f} ordens de grandeza)")
fs_mem = fatores_crescimento(df, MITM, "com_solucao", "mem_mb")
regime = [f for a, f in fs_mem if a >= 25]
print("Fator de crescimento da memória do MITM a cada +5 em n:\n  "
      + ", ".join(f"{a - 5}→{a}: {f:.1f}x" for a, f in fs_mem)
      + (f"  |  média geométrica (n≥25): {geometric_mean(regime):.1f}x" if regime else ""))
if 50 not in com_mem and 45 in com_mem and regime:
    print(f"Pico projetado para n=50: {fmt_mem(st_mem[45]['mem_mb'] * geometric_mean(regime))}")

# ------------------------------------------------------------------
secao("EXPERIMENTO 3 (Seção 5.3) — efeito da poda (instâncias com solução)")
print(f"{'n':>3} | {'nós sem poda':>15} | {'nós com poda':>15} | {'fator':>7} | obs.")
fatores_poda = []
for n in NS:
    c = stats(df_poda, BT, "com_solucao", n)
    s = stats(df_poda, BT_SEM_PODA, "com_solucao", n)
    if c["total"] == 0 and s["total"] == 0:
        continue
    col_sem = "T.L." if eh_tl(s) else ("—" if s["total"] == 0 else f"{s['estados']:,.0f}")
    col_com = "—" if c["total"] == 0 else f"{c['estados']:,.0f}"
    fator = "—"
    if s["total"] and c["total"] and not eh_tl(s) and not eh_tl(c):
        fatores_poda.append((n, s["estados"] / c["estados"]))
        fator = f"{fatores_poda[-1][1]:.0f}x"
    obs = (f"sem poda: {s['timeouts']}/{s['total']} T.L." if 0 < s["timeouts"] < s["total"] else "")
    print(f"{n:>3} | {col_sem:>15} | {col_com:>15} | {fator:>7} | {obs}")
if fatores_poda:
    n_f, f_f = max(fatores_poda, key=lambda x: x[1])
    print(f"Fator máximo de redução da poda: {f_f:.0f}x em n={n_f}")
tl_sem_poda = [n for n in NS if eh_tl(stats(df_poda, BT_SEM_PODA, "com_solucao", n))]
if tl_sem_poda:
    print(f"Primeiro n em que TODAS as repetições sem poda estouram: n={min(tl_sem_poda)}")
st_poda = {n: stats(df_poda, BT, "com_solucao", n) for n in NS}
validos_poda = {n: s for n, s in st_poda.items() if s["total"] and not eh_tl(s)}
n_max_nos = max(validos_poda, key=lambda k: validos_poda[k]["estados"])
print(f"Máximo de nós do BT com poda: {validos_poda[n_max_nos]['estados']:,.0f} em n={n_max_nos} "
      f"(até n={max(validos_poda)})")

# ------------------------------------------------------------------
secao("EXPERIMENTO 4 (Seção 5.4) — sensibilidade à entrada")
bt50 = stats(df, BT, "alvo_pequeno", 50)
bt45, m45 = stats(df, BT, "alvo_pequeno", 45), stats(df, MITM, "alvo_pequeno", 45)
print(f"Alvo pequeno — BT em n=50: {fmt_tempo(bt50)}")
print(f"Alvo pequeno — n=45: BT = {fmt_t(bt45['tempo'])} vs MITM = {fmt_t(m45['tempo'])}"
      f"  →  BT {m45['tempo'] / bt45['tempo']:.0f}x mais rápido")

st_bt = {n: stats(df, BT, "com_solucao", n) for n in NS}
validos = {n: s for n, s in st_bt.items() if s["total"] and not eh_tl(s)}
n_pico = max(validos, key=lambda k: validos[k]["tempo"])
pico = validos[n_pico]
print(f"Com solução — pico de tempo do BT: {fmt_t(pico['tempo'])} em n={n_pico} "
      f"({pico['estados']:,.0f} nós em média)")
apos = {n: s for n, s in validos.items() if n > n_pico}
if apos:
    ts = [s["tempo"] for s in apos.values()]
    es = [s["estados"] for s in apos.values()]
    print(f"Com solução — após o pico (n>{n_pico}): tempo entre {fmt_t(min(ts))} e {fmt_t(max(ts))}, "
          f"nós entre {min(es):,.0f} e {max(es):,.0f}")

disp = []
for n in NS:
    vals, cens = [], False
    for t in TIPOS:
        s = stats(df, BT, t, n)
        if s["total"] == 0:
            vals = []
            break
        # ponto totalmente estourado entra como limite inferior TIMEOUT_S
        vals.append(TIMEOUT_S if eh_tl(s) else s["tempo"])
        cens = cens or eh_tl(s)
    if vals:
        disp.append((n, max(vals) / min(vals), cens))
n_d, f_d, cens = max(disp, key=lambda x: x[1])
print(f"Dispersão do BT entre classes (mesmo n): máx. {'≥' if cens else ''}{f_d:,.0f}x em n={n_d} "
      f"(~{log10(f_d):.1f} ordens de grandeza)")

est = []
for n in NS:
    ss = [stats(df, MITM, t, n) for t in TIPOS]
    if all(s["total"] and not eh_tl(s) for s in ss):
        ts = [s["tempo"] for s in ss]
        est.append((n, max(ts) / min(ts)))
if est:
    n_e, f_e = max(est, key=lambda x: x[1])
    print(f"Estabilidade do MITM entre classes: variação máxima de {f_e:.1f}x (em n={n_e})")

valor_medio = (VALOR_MIN_CENT + VALOR_MAX_CENT) / 2
alvo_medio = lambda n: n / 2 * valor_medio  # classe com_solucao usa ~metade dos elementos
n_dens = next(n for n in range(10, 61) if 2 ** n >= alvo_medio(n))
print(f"Transição de densidade (2^n ≥ alvo médio ≈ (n/2)·{valor_medio:,.0f} centavos): n ≈ {n_dens}  "
      f"(2^{n_dens} = {2 ** n_dens:.2g} vs alvo médio {alvo_medio(n_dens):.2g})")

# ------------------------------------------------------------------
secao("SEÇÃO 5.5 — linhas efetivas de código (exclui vazias, comentários e docstrings)")

def linhas_efetivas(path):
    src = Path(path).read_text()
    doc = set()
    for node in ast.walk(ast.parse(src)):
        corpo = getattr(node, "body", None)
        if (isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
                and corpo and isinstance(corpo[0], ast.Expr)
                and isinstance(corpo[0].value, ast.Constant)
                and isinstance(corpo[0].value.value, str)):
            doc.update(range(corpo[0].lineno, corpo[0].end_lineno + 1))
    return sum(1 for i, linha in enumerate(src.splitlines(), 1)
               if linha.strip() and not linha.strip().startswith("#") and i not in doc)

for mod in ["utils/backtracking.py", "utils/meet_in_middle.py"]:
    print(f"{mod}: {linhas_efetivas(mod)} linhas efetivas")